# 原SOXL短期收益的直接反转排序：对f696配置意图的固定必要门

原39项既有soxl_ret3只按训练负号取反；同940日对照f696提案，完整年/季度/819窗口与39项筛选修正区块门。未拟合或改变账户。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import gzip,hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from sklearn.metrics import roc_auc_score
root=Path.cwd();card=root/'research/stage1_soxl_ret3_pairwise_incremental_rank_20261024';plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());s=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_market_GETs','new_model_fits','new_native_accounts','new_Recorders'))
for path,key in [(root/s['audit_card']/'plan.json','audit_plan_sha256'),(root/s['audit_card']/'result.json','audit_result_sha256'),(root/s['dataset_card']/'summary.json','dataset_summary_sha256'),(root/s['timing_card']/'summary.json','timing_summary_sha256'),(root/s['f696_card']/'summary.json','f696_summary_sha256'),(root/s['account_card']/'summary.json','account_summary_sha256'),(root/s['provider']/'qlib_store_manifest.json','provider_manifest_sha256')]:assert sha(path)==s[key],key
audit=json.loads((root/s['audit_card']/'result.json').read_text());assert 'soxl_ret3' in audit['train_selected_top5']
a=root/s['dataset_card'];t=root/s['timing_card'];f=root/s['f696_card'];b=root/s['account_card']
sa=json.loads((a/'summary.json').read_text());sf=json.loads((f/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text())
assert sa['nas_readback']['verified'] and sa['nas_readback']['sha256']==s['dataset_NAS_sha256']
ra=json.loads((f/'readback_verification.json').read_text());assert ra['status']=='passed' and ra['all_artifact_hashes_match'] and ra['archive_sha256']==s['f696_NAS_sha256']
assert sb['nas_readback']['verified'] and sb['nas_readback']['sha256']==s['account_NAS_sha256']
for source,exp,rid,art,key in [(a,'dataset_experiment_id','dataset_recorder_id','inputs/dataset','dataset_artifact_sha256'),(t,'timing_experiment_id','timing_recorder_id','inputs/timing','timing_artifact_sha256'),(f,'f696_experiment_id','f696_recorder_id','inputs/frame','f696_frame_sha256'),(b,'account_experiment_id','account_recorder_id','analysis/rolling126','account_rolling_sha256')]:assert sha(source/'qlib_recorder'/s[exp]/s[rid]/'artifacts'/art)==s[key],key
qlib.init(provider_uri=str((root/s['provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((a/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['dataset_experiment_id'],recorder_id=s['dataset_recorder_id']);dataset=rec.load_object('inputs/dataset')
with R.uri_context(str((t/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['timing_experiment_id'],recorder_id=s['timing_recorder_id']);timing=rec.load_object('inputs/timing')
with R.uri_context(str((f/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['f696_experiment_id'],recorder_id=s['f696_recorder_id']);frame=rec.load_object('inputs/frame').sort_values('trade_date').reset_index(drop=True)
with R.uri_context(str((b/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['account_experiment_id'],recorder_id=s['account_recorder_id']);rolling=rec.load_object('analysis/rolling126')
tr_full=dataset.prepare('train',col_set=['feature','label','meta']);he_full=dataset.prepare('held',col_set=['feature','label','meta'])
tr=tr_full.loc[tr_full[('meta','train_mature')].astype(bool)];he=he_full.loc[he_full[('meta','held_mature')].astype(bool)]
assert len(tr)==3014 and len(he)==940 and len(frame)==944 and len(rolling)==819
assert len(tr['feature'].columns)==39 and 'soxl_ret3' in tr['feature'].columns
train_x=tr[('feature','soxl_ret3')].astype(float);train_margin=tr[('label','SOXL_relative_H5')].astype(float)-tr[('label','cash_advantage_H5')].astype(float)
train_rho=float(train_x.corr(train_margin,method='spearman'));audit_rho=next(z['train']['spearman'] for z in audit['all39'] if z['feature']=='soxl_ret3')
assert abs(train_rho-audit_rho)<1e-12 and train_rho<0
held_dates=pd.DatetimeIndex(he.index.get_level_values('datetime'));full_dates=pd.DatetimeIndex(he_full.index.get_level_values('datetime'))
assert full_dates.equals(pd.DatetimeIndex(frame.signal_date)) and held_dates.equals(full_dates[:940])
assert timing.index.is_unique and full_dates.isin(timing.index).all()
decisions=pd.to_datetime(timing.loc[full_dates,'decision_at'],utc=True).dt.tz_convert('America/New_York').dt.date
assert all(z>d.date() for z,d in zip(decisions,full_dates))
held_margin=he[('label','SOXL_relative_H5')].to_numpy(float)-he[('label','cash_advantage_H5')].to_numpy(float)
held_score=-he[('feature','soxl_ret3')].to_numpy(float);control=(frame.soxl_weight-frame.cash_weight).to_numpy(float)[:940]
assert np.isfinite(np.column_stack([held_margin,held_score,control])).all()
assert int((abs(held_margin)<=1e-12).sum())==0
y=(held_margin>0).astype(int)
def metrics(idx):
 yy=y[idx];a=held_score[idx];c=control[idx]
 return {'rows':len(yy),'positive':int(yy.sum()),'candidate_auc':float(roc_auc_score(yy,a)) if len(set(yy))==2 else None,
         'f696_auc':float(roc_auc_score(yy,c)) if len(set(yy))==2 else None,
         'candidate_score_min_max':[float(a.min()),float(a.max())]}
full=metrics(np.arange(940));assert abs(full['f696_auc']-.46799513276178756)<1e-12
annual=[{'year':int(year),'partial_endpoint':year in (2023,2026),**metrics(np.flatnonzero(held_dates.year==year))} for year in (2023,2024,2025,2026)]
quarter=[]
for year in (2023,2024,2025,2026):
 for q in (1,2,3,4):
  idx=np.flatnonzero((held_dates.year==year)&(held_dates.quarter==q))
  if len(idx):quarter.append({'year':year,'quarter':q,**metrics(idx)})
halves=[{'start':'2023-01-03','end':'2024-12-31',**metrics(np.arange(502))},{'start':'2025-01-02','end':'2026-10-07',**metrics(np.arange(502,940))}]
assert len(quarter)==16 and halves[0]['rows']==502 and halves[1]['rows']==438
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'))
close=D.features(['SOXX'],['$close'],start_time='2011-01-03',end_time='2026-10-09',freq='day')['$close'].droplevel('instrument').sort_index()
assert close.index.equals(calendar)
windows=[]
for i in range(819):
 rw=rolling.iloc[i];g=frame.iloc[i:i+126];assert pd.Timestamp(g.trade_date.iloc[0])==pd.Timestamp(rw.first_trade)
 pos=calendar.get_loc(pd.Timestamp(g.signal_date.iloc[0]));assert pos>=20
 r5=float(close.iloc[pos]/close.iloc[pos-5]-1);r20=float(close.iloc[pos]/close.iloc[pos-20]-1)
 state='uptrend' if r5>=0 and r20>=0 else 'rebound' if r5>=0 else 'pullback' if r20>=0 else 'decline'
 z=metrics(np.arange(i,min(i+126,940)))
 windows.append({'start_trade':str(pd.Timestamp(rw.first_trade).date()),'year':int(pd.Timestamp(rw.first_trade).year),'start_state':state,
                 'outcome':'weak' if rw.relative_log_excess<0 else 'good' if rw.relative_log_excess>0 else 'tie',
                 'mature':z['rows'],'candidate_auc':z['candidate_auc'],'f696_auc':z['f696_auc']})
w=pd.DataFrame(windows);assert w.outcome.value_counts().to_dict()=={'weak':564,'good':255} and w.mature.min()>=122
cells=[]
for (state,year),g in w.groupby(['start_state','year'],sort=True):
 weak=g[g.outcome.eq('weak')];good=g[g.outcome.eq('good')]
 row={'start_state':state,'year':int(year),'weak_windows':len(weak),'good_windows':len(good),'matched':bool(len(weak) and len(good))}
 if row['matched']:row['weak_minus_good_auc_improvement']=float((weak.candidate_auc-weak.f696_auc).mean()-(good.candidate_auc-good.f696_auc).mean())
 cells.append(row)
assert len([z for z in cells if z['matched']])==12 and all(z['good_windows']==0 for z in cells if z['year']==2023)
rng=np.random.default_rng(20261024);reps=np.empty(10000,float);valid=0
for i in range(10000):
 indices=[]
 while len(indices)<944:
  start=int(rng.integers(0,944));length=int(rng.geometric(1/63));indices.extend((start+np.arange(length))%944)
 idx=np.asarray(indices[:944],int);idx=idx[idx<940]
 if len(set(y[idx]))<2:reps[i]=np.nan;continue
 reps[i]=float(roc_auc_score(y[idx],held_score[idx])-roc_auc_score(y[idx],control[idx]));valid+=1
assert valid>=9900
alpha=.025/39;lower=float(np.nanquantile(reps,alpha));upper=float(np.nanquantile(reps,1-alpha))
bootstrap={'resamples':10000,'valid':valid,'mean_block':63,'calendar_rows':944,'scored_rows':940,'one_sided_alpha':alpha,
           'paired_auc_improvement':full['candidate_auc']-full['f696_auc'],'corrected_lower':lower,'upper':upper}
gates={'full_auc_above_random':full['candidate_auc']>.5,'full_improvement_above_0p02':full['candidate_auc']>full['f696_auc']+.02,
       '2023_2024_2025_auc_above_random':all(z['candidate_auc']>.5 for z in annual if z['year'] in (2023,2024,2025)),
       'both_halves_auc_above_random':all(z['candidate_auc']>.5 for z in halves),'corrected_block_lower_positive':lower>0}
decision='conditional_existing_short_reversal_rank_pass_requires_incremental_economic_receiver' if all(gates.values()) else 'not_promoted_fixed_soxl_ret3_pairwise_incremental_rank_failed'
result={'decision':decision,'plan_sha256':sha(plan_path),'source_recorder_ids':{'dataset':s['dataset_recorder_id'],'timing':s['timing_recorder_id'],'f696':s['f696_recorder_id'],'account':s['account_recorder_id']},
        'train_mature_rows':len(tr),'held_mature_rows':len(he),'full_source_rows':len(frame),'unknown_last_four':4,'train_soxl_ret3_spearman':train_rho,
        'full':full,'annual':annual,'quarter':quarter,'halves':halves,'rolling126_windows':len(w),'matched_cells':cells,
        'bootstrap':bootstrap,'gates':gates,'information_necessary_pass':all(gates.values()),
        'new_market_GETs':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':1,'new_Recorders':0,
        'independent_validation':False,'SOXX_goal_achieved':False,
        'limits':['This original39 coordinate was selected after extensive reused-history research, so historical AUC is development feedback only','Gross H5 future price sign is not a funded fee-net Stage1 action label','A one-dimensional rank pass is insufficient for a legal joint allocation and risk-cap account']}
window_blob=gzip.compress(json.dumps(windows,ensure_ascii=False,separators=(',',':')).encode(),mtime=0)
win_path=card/'rolling126.json.gz';win_path.write_bytes(window_blob);result['rolling126_gzip_path']=win_path.name;result['rolling126_gzip_sha256']=sha(win_path)
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'decision':decision,'train_rho':train_rho,'full':full,'annual':[(z['year'],z['candidate_auc'],z['f696_auc']) for z in annual],'halves':[(z['candidate_auc'],z['f696_auc']) for z in halves],'bootstrap':bootstrap,'gates':gates})


{'decision': 'not_promoted_fixed_soxl_ret3_pairwise_incremental_rank_failed', 'train_rho': -0.07037970952611793, 'full': {'rows': 940, 'positive': 539, 'candidate_auc': 0.5194920861112524, 'f696_auc': 0.46799513276178756, 'candidate_score_min_max': [-0.5084418058395386, 0.4526646137237549]}, 'annual': [(2023, 0.49099330931549146, 0.3757076685537828), (2024, 0.5630787037037038, 0.48662551440329216), (2025, 0.5069817400644469, 0.5067132116004297), (2026, 0.5058645918011846, 0.5054000696783184)], 'halves': [(0.5305273638232271, 0.42617869989722507), (0.5096001904885599, 0.4985605125873975)], 'bootstrap': {'resamples': 10000, 'valid': 10000, 'mean_block': 63, 'calendar_rows': 944, 'scored_rows': 940, 'one_sided_alpha': 0.000641025641025641, 'paired_auc_improvement': 0.05149695334946486, 'corrected_lower': -0.06244828403341826, 'upper': 0.2182694187013961}, 'gates': {'full_auc_above_random': True, 'full_improvement_above_0p02': True, '2023_2024_2025_auc_above_random': False, 'both_halves_au

## 中文结论

# 原SOXL三日收益的直接反转排序：2023与修正不确定性门失败

按上一卡3014个成熟2011–2022训练标签中已冻结的负号，只将原39项 `soxl_ret3` 取反，对相同940个成熟复用2023–2026五日SOXL胜现金毛价格标签排序；精确f696 09:29 `soxl_weight−cash_weight` 提案为同日对照。Qlib来源、时钟、39项训练序次、944提案日期、16季度与全部819个原账户126日窗口已核对，末4个标签未知；逐窗汇总保存在`rolling126.json.gz`。0新行情、拟合或经济账户，一次预定10000抽样区块检验。

训练期 `soxl_ret3` 对连续成对边际Spearman仅 **−0.07038**。评价全段负号分数AUC **0.51949**，f696提案 **0.467995**，点差 **+0.05150**；但2023部分年候选 **0.49099**（仍低于随机0.5），完整2024 **0.56308**，完整2025 **0.50698**（f696同期 **0.50671**），2026部分年 **0.50586**。固定前后半 **0.53053/0.50960**。考虑这项来自39项训练筛选，完整944日历平均块长63日、10000次抽样、单侧0.025/39的配对AUC增量下界 **−0.06245**。五项固定必要门 **3/5**，2023与修正下界失败。

2023原账户所有126日窗口均弱于SOXX，没有同年前置走势状态中的好窗口可比；按其弱段将短反转解释为可交易切换尤其缺证据。候选虽在全段点估计超过原配置排序，却没有证明相对原39项配置模型的稳定经济增量，更未证明风险容量、合法动作和费用后SOXX财富改善。该因子已经是Stage1输入，且评价历史曾用于提出假说。

决定 `not_promoted_fixed_soxl_ret3_pairwise_incremental_rank_failed`。不改三日跨度、符号、年份门或风险上限救援；不进入完整0bp候选账户。下一机制需要实质不同的决策前信息或经济接收任务，训练与合格前向严格分开。SOXX v2目标未达成。
